In [1]:
from collections import deque
import numpy as np
import argparse
import imutils
import cv2

In [2]:
import threading # 线程
import ctypes
import inspect
import time
import ipywidgets.widgets as widgets
from IPython.display import display

In [3]:
from arm5dof_uservo import Arm5DoFUServo
from arm_test_origin import ArmApplication
from config import *
# 创建机械臂
arm = Arm5DoFUServo(is_debug=True)
# 创建机械臂应用
arm_app = ArmApplication(arm, is_debug=True)

In [4]:
# ============ 显示控件 ============
def bgr8_to_jpeg(value, quality=75):
    return bytes(cv2.imencode('.jpg', value)[1])

Frame = widgets.Image(format='jpeg', width=600, height=480)
ColorLabel = widgets.Label(value="中心颜色: ...")
StateLabel = widgets.Label(value="状态: 初始化")
display(Frame, ColorLabel, StateLabel)

# ============ 摄像头 ============
camera = cv2.VideoCapture(0)

# ============ 颜色阈值 ============
COLOR_RANGES = {
    "red":    [((0, 60, 40),    (8, 255, 255)),      # H 上界 15→8
               ((170, 60, 40),  (180, 255, 255))],   # 下界 165→170
    "yellow": [((15, 50, 100),  (40, 255, 255))],    # H 下界 10→15，上界 45→40
    "green":  [((35, 50, 50),   (90, 255, 255))],
    "blue":   [((90, 50, 50),   (140, 255, 255))]
}
ROI_HALF = 25
_yy, _xx = np.mgrid[0:ROI_HALF*2, 0:ROI_HALF*2]
_c0 = ROI_HALF
_WEIGHT = np.exp(-(((_xx-_c0)**2 + (_yy-_c0)**2) / (2*(ROI_HALF*0.6)**2)))
_WEIGHT = (_WEIGHT / _WEIGHT.max()).astype(np.float32)

def detect_center_color(frame):
    h, w = frame.shape[:2]
    cx, cy = w // 2, h // 2
    roi = frame[cy-ROI_HALF:cy+ROI_HALF, cx-ROI_HALF:cx+ROI_HALF]
    if roi.size == 0:
        return None
    hsv = cv2.cvtColor(roi, cv2.COLOR_BGR2HSV).astype(np.float32)
    hsv[..., 1] = np.clip(hsv[..., 1] * 1.6, 0, 255)
    hsv[..., 2] = np.clip(hsv[..., 2] * 0.8, 0, 255)
    hsv = hsv.astype(np.uint8)

    best_color, best_score = None, 0.0
    for name, ranges in COLOR_RANGES.items():
        mask = np.zeros(hsv.shape[:2], dtype=np.uint8)
        for lower, upper in ranges:
            mask |= cv2.inRange(hsv, np.array(lower), np.array(upper))
        mask = cv2.erode(mask, None, iterations=1)
        mask = cv2.dilate(mask, None, iterations=1)
        score = float(np.sum(mask.astype(np.float32) * _WEIGHT))
        if score > best_score:
            best_score, best_color = score, name
    if best_score < float(np.sum(_WEIGHT)) * 0.03:
        return None
    return best_color

# ============ 共享状态 ============
current_color = None          # 检测线程实时颜色（仅观察阶段有效）
pending_color = None          # ★ 本轮已确认、待分类的颜色
color_lock = threading.Lock()

# ★ 颜色识别开关：只在观察阶段打开
detect_enabled = threading.Event()
detect_enabled.clear()

# ============ 检测线程（只负责"看"，不执行动作） ============
def Video_display():
    global current_color
    while True:
        grabbed, frame = camera.read()
        if not grabbed:
            continue
        frame = imutils.resize(frame, width=600)

        if detect_enabled.is_set():
            color = detect_center_color(frame)
        else:
            color = None

        with color_lock:
            current_color = color

        h, w = frame.shape[:2]
        cx, cy = w // 2, h // 2
        cv2.rectangle(frame, (cx-ROI_HALF, cy-ROI_HALF),
                      (cx+ROI_HALF, cy+ROI_HALF), (255, 0, 255), 2)
        cv2.putText(frame, color if color else "none",
                    (cx-ROI_HALF, cy-ROI_HALF-8),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 0, 255), 2)
        Frame.value = bgr8_to_jpeg(frame)

        if detect_enabled.is_set():
            ColorLabel.value = "中心颜色: " + (color if color else "无")
        else:
            ColorLabel.value = "中心颜色: (识别已暂停)"

# ============ 机械臂动作 ============

def action_A_observe():
    StateLabel.value = "状态: A 观察中..."
    print("[A] 机械臂回到观察位")
    arm_app.grab_move(-25, -1, 15)
    time.sleep(0.5)


def action_S():
    StateLabel.value = "状态: S 预处理"
    print("[S] 执行行动 S")
    arm_app.home()
    arm_app.grab_cubic(0, 0, 3)     # 抓取物块
    arm_app.home()                  # 回 home 位（物块保持夹住）


def action_B_red():
    StateLabel.value = "状态:有害垃圾"
    print("[B] 处理红色物块")
    arm_app.grab_move(-50, -125, 150)


def action_C_yellow():
    StateLabel.value = "状态:其他垃圾"
    print("[C] 处理黄色物块")
    arm_app.grab_move(45, 125, 150)


def action_D_green():
    StateLabel.value = "状态:厨余垃圾"
    print("[D] 处理绿色物块")
    arm_app.grab_move(-50, 125, 150)


def action_E_blue():
    StateLabel.value = "状态: 可回收垃圾"
    print("[E] 处理蓝色物块")
    arm_app.grab_move(45, -125, 150)


# 颜色 → 动作 的映射表
COLOR_ACTION = {
    "red":    action_B_red,
    "yellow": action_C_yellow,
    "green":  action_D_green,
    "blue":   action_E_blue,
}


S_INIT      = "INIT"
S_OBSERVE   = "OBSERVE"
S_ACTION    = "ACTION"
S_IDLE      = "IDLE"

CONFIRM_FRAMES = 5
OBSERVE_TIMEOUT = 10.0
OBSERVE_DWELL = 2.0
CYCLE_LIMIT =8

def get_color():
    with color_lock:
        return current_color

def wait_stable_color(timeout):
    t0 = time.time()
    last = object()
    cnt = 0
    while time.time() - t0 < timeout:
        c = get_color()
        if c == last:
            cnt += 1
            if cnt >= CONFIRM_FRAMES:
                return c
        else:
            last = c
            cnt = 1 if c is not None else 0
        time.sleep(0.05)
    return None

def state_machine():
    global pending_color          # ★ 声明可修改全局

    state = S_INIT
    cycle = 0

    while True:
        # ---------- INIT ----------
        if state == S_INIT:
            print("==== 系统启动 ====")
            state = S_OBSERVE

        # ---------- OBSERVE ----------
        elif state == S_OBSERVE:
            detect_enabled.clear()

            action_A_observe()

            detect_enabled.set()
            StateLabel.value = "状态: 观察中（%.0f秒）" % OBSERVE_DWELL
            print("[观察] 开启识别，停留 %.1f 秒" % OBSERVE_DWELL)
            time.sleep(OBSERVE_DWELL)

            color = wait_stable_color(OBSERVE_TIMEOUT)

            detect_enabled.clear()
            print("[识别] 关闭")

            if color is None:
                StateLabel.value = "状态: 观察超时，重试"
                print("[!] 未识别到有效颜色，重新观察")
                state = S_OBSERVE
            else:
                print("[✓] 识别为:", color)

                pending_color = color       # ★ 保存本轮颜色
                action_S()
                state = S_ACTION

        # ---------- ACTION ----------
        elif state == S_ACTION:
            color = pending_color           # ★ 用保存的颜色，不用 current_color

            action_fn = COLOR_ACTION.get(color)
            if action_fn is None:
                print("[!] 颜色无效，跳过")
                pending_color = None
                state = S_OBSERVE
                continue

            try:
                action_fn()
            except Exception as e:
                print("[!] 动作异常:", e)
                pending_color = None
                state = S_OBSERVE
                continue

            pending_color = None            # ★ 本轮完成清空
            cycle += 1
            print("[✓] 第 %d 轮完成" % cycle)

            if CYCLE_LIMIT is not None and cycle >= CYCLE_LIMIT:
                state = S_IDLE
            else:
                state = S_OBSERVE

        # ---------- IDLE ----------
        elif state == S_IDLE:
            detect_enabled.clear()
            StateLabel.value = "状态: 全部完成"
            print("==== 所有任务完成 ====")
            time.sleep(0.2)

        time.sleep(0.02)

# ============ 启动 ============
t_detect = threading.Thread(target=Video_display, daemon=True)
t_sm     = threading.Thread(target=state_machine, daemon=True)
t_detect.start()
t_sm.start()

Image(value=b'', format='jpeg', height='480', width='600')

Label(value='中心颜色: ...')

Label(value='状态: 初始化')

==== 系统启动 ====
[A] 机械臂回到观察位
运动到 wx=-25.00, wy=-1.00, wz=70.00
运动到 arm_x=151.93, arm_y=-2.18, wz=54.69
运动到 wx=-25.00, wy=-1.00, wz=20.00
运动到 arm_x=151.93, arm_y=-2.18, wz=4.69
[观察] 开启识别，停留 2.0 秒
[识别] 关闭
[✓] 识别为: blue
[S] 执行行动 S
运动到 wx=0.00, wy=0.00, wz=58.00
运动到 arm_x=176.91, arm_y=-0.78, wz=42.69
运动到 wx=0.00, wy=0.00, wz=8.00
运动到 arm_x=176.91, arm_y=-0.78, wz=-7.31
运动到 wx=0.00, wy=0.00, wz=73.00
运动到 arm_x=176.91, arm_y=-0.78, wz=57.69
[E] 处理蓝色物块
运动到 wx=45.00, wy=-125.00, wz=205.00
运动到 arm_x=223.91, arm_y=-125.06, wz=189.69
运动到 wx=45.00, wy=-125.00, wz=155.00
运动到 arm_x=223.91, arm_y=-125.06, wz=139.69
[✓] 第 1 轮完成
[A] 机械臂回到观察位
运动到 wx=-25.00, wy=-1.00, wz=70.00
运动到 arm_x=151.93, arm_y=-2.18, wz=54.69
运动到 wx=-25.00, wy=-1.00, wz=20.00
运动到 arm_x=151.93, arm_y=-2.18, wz=4.69
[观察] 开启识别，停留 2.0 秒
[识别] 关闭
[✓] 识别为: green
[S] 执行行动 S
运动到 wx=0.00, wy=0.00, wz=58.00
运动到 arm_x=176.91, arm_y=-0.78, wz=42.69
运动到 wx=0.00, wy=0.00, wz=8.00
运动到 arm_x=176.91, arm_y=-0.78, wz=-7.31
运动到 wx=0.00, wy=0.00, wz=7